# Chapter 23: Discounting and Risk Margins

Code of Chapter 23 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## Exercise 1: Long tail

In [ ]:
from gidiscount import present_value
import pandas as pd
present_value(pd.Series(100_000.0, index=range(2026, 2036)), 2025, 0.04)   # 827152

## 23.6 Python Walkthrough

In [ ]:
import numpy as np
import pandas as pd
from gitriangle import triangle, synthetic_history
from givar import odp_bootstrap
from gidiscount import (future_cash_flows, discount_factors, present_value,
                        cost_of_capital_margin, confidence_level)

_, claims, tx = synthetic_history()
pt = triangle(claims, tx, "paid", 2021, 2025)
cf = future_cash_flows(pt, 2025)                          # 1581053, 468759, 91639, 21441
present_value(cf, 2025, 0.03)                             # 2110732
present_value(cf, 2025, [0.025, 0.028, 0.030, 0.032])     # 2115706

sims = odp_bootstrap(pt, 10_000, seed=22)["reserves"]
scr0 = np.quantile(sims, 0.995) - sims.mean()             # 703474
rm = cost_of_capital_margin(cf, scr0, 2025, coc=0.06, rate=0.03)["margin"]   # 54073
lic = present_value(cf, 2025, 0.03) + rm
confidence_level(sims * present_value(cf, 2025, 0.03) / cf.sum(), lic)       # 0.587